In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path
from collections import defaultdict
import re

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import statsmodels.stats.contingency_tables as sm

from utils.plots import calculate_metrics, calculate_accuracy
from utils.tools import get_config, collect_jobs, prettify_table, format_table, aggregate_results
from utils.constants import PRED_COLS

In [3]:
def format_model_name(name):
    parts = name.split("-")
    match = re.search(r"\d{1,2}B", name)
    return f"{parts[0]}-{match.group(0)}" if match else name

def get_jobs_per_model_and_suite(jobs):
    
    print(f"Processing {len(jobs)} jobs.")
    # Dict of model -> suite -> seed -> jobId
    table = defaultdict(lambda: defaultdict(lambda: defaultdict(str)))
    for job in jobs:
        config = get_config(job["config_json"])
        model = config.get("model").split("/")[1]
        suite = get_suite(config)
        seed = config["seed"]

        table[model][suite][seed] = config.get("jobid")

    return table

def get_suite(cfg):

    #print(cfg)
    
    n_demos = cfg["number_of_demonstrations"]

    type_demos = (
        "non" 
        if n_demos == 0
        else "neg" 
        if cfg["type_of_demonstrations"] == -1
        else "mix" 
        if cfg["type_of_demonstrations"] == 0
        else "pos"
    )
    instr = (
        "impl" 
        if cfg["use_instructions"] == 0
        else "expl"
    )

    return f"{n_demos}-{type_demos}-{instr}"

In [4]:
def align_dataframes(df1, df2, key_cols=("title", "theme", "topic", "concept", "The exercise description matched the selected theme (Yes/No)", "The exercise description matched the selected topic (Yes/No)", "Included concepts that were too advanced (Yes/No)")):
    # Inner join on key columns to find common rows
    common_keys = pd.merge(
        df1[list(key_cols)].drop_duplicates(),
        df2[list(key_cols)].drop_duplicates(),
        on=list(key_cols),
        how="inner"
    )

    # Keep only matching rows
    df1_aligned = df1.merge(common_keys, on=list(key_cols), how="inner")
    df2_aligned = df2.merge(common_keys, on=list(key_cols), how="inner")

    # Sort identically so row order matches
    df1_aligned = df1_aligned.sort_values(list(key_cols)).reset_index(drop=True)
    df2_aligned = df2_aligned.sort_values(list(key_cols)).reset_index(drop=True)

    return df1_aligned, df2_aligned


def get_model_family(model):
    match model.split("-")[0]:
        case "Qwen3":
            return "Qwen"
        case "Mistral":
            return "mistralai"
        case "Llama":
            return "meta-llama"
        case _:
            return "default"


def get_csv(model, jobid):
    fam = get_model_family(model)
    return f"./outputs/v6/{fam}/{model}/{jobid}/result.csv"


def get_errored_rows(df):
    return ~df[PRED_COLS].isin(["yes", "no"]).all(axis=1)


def normalize_df(df):
    return df.map(lambda val:
        "yes"
        if val == "\"yes\""
        else "no"
        if val == "\"no\""
        else val
    )

def rename_result_cols(df, prefix="baseline"):
    return df.rename(columns={
        0: f"{prefix}_theme_prediction_correct",
        1: f"{prefix}_topic_prediction_correct",
        2: f"{prefix}_concept_prediction_correct"
    })
    
def get_paired_predictions(jobs, model, baseline, against):
    # Mapping seed -> jobid
    jobids_base = jobs[model][baseline]
    jobids_against = jobs[model][against]

    # Append all predictions to these dataframes for pairwise calculations
    preds_base = None
    preds_against = None   

    # Calculate pairwise differences
    for seed in jobids_base.keys():
        jobid_bl = jobids_base[seed]
        jobid_ag = jobids_against[seed]
      
        base = normalize_df(pd.read_csv(get_csv(model, jobid_bl), sep=";"))      
        comp = normalize_df(pd.read_csv(get_csv(model, jobid_ag), sep=";"))

        # We need to drop the rows that were used as demonstrations from the baseline,
        # Because we can't calculate the pairwise connections for these rows
        base, comp = align_dataframes(base, comp)

        # Since both dataframes may contain errors and on different samples,
        # need to remove erroneous samples from both dataframes
        error_mask = (get_errored_rows(base) | get_errored_rows(comp))
        base = base.loc[~error_mask]
        comp = comp.loc[~error_mask]

        # Next, get correct/incorrect predictions
        base_correct = pd.DataFrame(base[GT_COLS].values == base[PRED_COLS].values)
        comp_correct = pd.DataFrame(comp[GT_COLS].values == comp[PRED_COLS].values)

        # Append predictions
        if preds_base is None:
            preds_base = base_correct.copy()
            preds_against = comp_correct.copy()
        else:
            preds_base = pd.concat([preds_base, base_correct])
            preds_against = pd.concat([preds_against, comp_correct])

    # All predictions have been collected into two dataframes in correct order
    return (
        rename_result_cols(preds_base.reset_index(drop=True), baseline),
        rename_result_cols(preds_against.reset_index(drop=True), against)
    )

# Use result of get_paired_predictions
def build_contingency_tables(df1, df2):
    return [pd.crosstab(df1[col1], df2[col2]) for col1, col2 in zip(df1.columns, df2.columns)]

# Use result of build_contingency_tables
def my_mcnemar(tables, e=True, c=False):
    return [sm.mcnemar(tab, exact=e, correction=c) for tab in tables]

# Use result df of get_paired_predictions
def my_accuracy(df):
    counts = df.apply(lambda s: s.value_counts())   
    if True not in counts.index:
        return 0
    return counts.loc[True] / len(df)
    

In [5]:
models = ["Qwen3-32B", "Qwen3-8B", "Mistral-7B-Instruct-v0.3", "Mistral-Small-3.2-24B-Instruct-2506", "Llama-3.1-8B-Instruct", "Llama-3.3-70B-Instruct"]
GT_COLS = ["The exercise description matched the selected theme (Yes/No)", "The exercise description matched the selected topic (Yes/No)", "Included concepts that were too advanced (Yes/No)"]

model_map = {
    format_model_name(model): model for model in models
}

In [6]:
# Collect raw data
basepath = "./outputs/v6"
finished_jobs = collect_jobs(basepath)
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten

jobs_grouped = get_jobs_per_model_and_suite(jobs_list)

Processing 330 jobs.


In [7]:
#jobs_grouped

In [8]:
# Builds a label section for one row
def build_label_section(accuracy, accuracy_bl, contingency_table, raw_mcnemar, corr_mcnemar, prefix):
    columns = ["Accuracy", "\\Delta", "n01", "n10", "McNemar \\Phi^2", "Raw p", "Corrected p", "Significance"]
    return {
        f"{prefix} Acc": accuracy,
        f"{prefix} $\\Delta$": accuracy - accuracy_bl,
        f"{prefix} n01": contingency_table.loc[False, True],
        f"{prefix} n10": contingency_table.loc[True, False],
        f"{prefix} $\\chi^2$": raw_mcnemar.statistic,
        f"{prefix} $p$": raw_mcnemar.pvalue,
        f"{prefix} $p^\prime$": corr_mcnemar.pvalue,
        f"{prefix} S": (
            "***"
            if corr_mcnemar.pvalue <= 0.001 
            else "**"
            if corr_mcnemar.pvalue <= 0.01
            else "*"
            if corr_mcnemar.pvalue <= 0.05
            else "ns"
        )
    }

def build_table_row(accuracies, accuracies_bl, contingency_tables, raw_mcnemars, corr_mcnemars):

    # {k: v for d in L for k, v in d.items()}
    prefixes = ["Theme", "Topic", "Concept"]
    return {
        col_name : value
        for section
        in [
            build_label_section(
                accuracies.iloc[i],
                accuracies_bl.iloc[i],
                contingency_tables[i],
                raw_mcnemars[i],
                corr_mcnemars[i],
                prefixes[i]
            )
            for i in range(3) 
        ]
        for col_name, value in section.items()
    }

# Builds table with accuracy, delta accuracy (to baseline), n01, n10, McNemar phi, raw p, corrected p, significance
# for one model
def build_mcnemar_table(jobs, model, baseline="0-non-expl"):
    model_jobs = jobs[model]

    rows = {}
    
    for suite in model_jobs.keys():
        #if suite == baseline:
        #    continue

        bl, ag = get_paired_predictions(jobs, model, baseline, suite)
        contingency_tables = build_contingency_tables(bl, ag)
        raw_mcnemars = my_mcnemar(contingency_tables, e=True)
        corr_mcnemars = my_mcnemar(contingency_tables, e=False, c=True)
        bl_accuracies = my_accuracy(bl)
        ag_accuracies = my_accuracy(ag)

        # For each label
        row = build_table_row(
            ag_accuracies,
            bl_accuracies,
            contingency_tables,
            raw_mcnemars,
            corr_mcnemars
        )
        rows[suite] = row
    
    return rows

In [9]:
mcnemar_tables = {model : build_mcnemar_table(jobs_grouped, fullname) for model, fullname in model_map.items()}
reform = {(outerKey, innerKey): values for outerKey, innerDict in mcnemar_tables.items() for innerKey, values in innerDict.items()}
mcnemar_table = pd.DataFrame.from_dict(reform, orient="index")
mcnemar_table.index = mcnemar_table.index.set_names(["Model", "Suite"])
mcnemar_table.columns = pd.MultiIndex.from_tuples(
    [
        (
            col[ : col.index(" ")],
            col[col.index(" ") + 1 : ]
        ) 
        for col in mcnemar_table.columns
    ]
)

mcnemar_table = mcnemar_table.sort_values(by=["Model", "Suite"])

/appl/scibuilder-mamba/aalto-rhel9/prod/software/scicomp-python-env/2025.2/b4b5f8e/lib/python3.12/site-packages/statsmodels/stats/contingency_tables.py:1348: RuntimeWarning: divide by zero encountered in scalar divide
  statistic = (np.abs(n1 - n2) - corr)**2 / (1. * (n1 + n2))
/appl/scibuilder-mamba/aalto-rhel9/prod/software/scicomp-python-env/2025.2/b4b5f8e/lib/python3.12/site-packages/statsmodels/stats/contingency_tables.py:1348: RuntimeWarning: divide by zero encountered in scalar divide
  statistic = (np.abs(n1 - n2) - corr)**2 / (1. * (n1 + n2))
/appl/scibuilder-mamba/aalto-rhel9/prod/software/scicomp-python-env/2025.2/b4b5f8e/lib/python3.12/site-packages/statsmodels/stats/contingency_tables.py:1348: RuntimeWarning: divide by zero encountered in scalar divide
  statistic = (np.abs(n1 - n2) - corr)**2 / (1. * (n1 + n2))
/appl/scibuilder-mamba/aalto-rhel9/prod/software/scicomp-python-env/2025.2/b4b5f8e/lib/python3.12/site-packages/statsmodels/stats/contingency_tables.py:1348: Runti

In [10]:
#mcnemar_table

In [11]:
print(mcnemar_table.to_latex(float_format="%.3g", column_format="ll|rrrrrrrl|rrrrrrrl|rrrrrrrl"))

\begin{tabular}{ll|rrrrrrrl|rrrrrrrl|rrrrrrrl}
\toprule
 &  & \multicolumn{8}{r}{Theme} & \multicolumn{8}{r}{Topic} & \multicolumn{8}{r}{Concept} \\
 &  & Acc & $\Delta$ & n01 & n10 & $\chi^2$ & $p$ & $p^\prime$ & S & Acc & $\Delta$ & n01 & n10 & $\chi^2$ & $p$ & $p^\prime$ & S & Acc & $\Delta$ & n01 & n10 & $\chi^2$ & $p$ & $p^\prime$ & S \\
Model & Suite &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{11}{*}{Llama-70B} & 0-non-expl & 0.942 & 0 & 0 & 0 & 0 & 1 & 0 & *** & 0.978 & 0 & 0 & 0 & 0 & 1 & 0 & *** & 0.85 & 0 & 0 & 0 & 0 & 1 & 0 & *** \\
 & 1-neg-expl & 0.893 & -0.0487 & 31 & 161 & 31 & 2.24e-22 & 1.28e-20 & *** & 0.926 & -0.0517 & 20 & 158 & 20 & 8.25e-28 & 9.76e-25 & *** & 0.828 & -0.0228 & 40 & 101 & 40 & 2.87e-07 & 4.35e-07 & *** \\
 & 1-neg-impl & 0.917 & -0.0259 & 22 & 91 & 22 & 3.7e-11 & 1.59e-10 & *** & 0.949 & -0.0285 & 15 & 91 & 15 & 1.9e-14 & 3.23e-13 & *** & 0.689 & -0.161 & 233 & 663 & 233 & 1.81e-48 & 1.38e-46 & *